In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage.io
import skimage.measure
from IPython.display import display, Markdown
import os

In [25]:
# Introduction
display(Markdown("""
# 📌 Coordinate Transformation Notebook

This notebook helps users transform coordinates from a global reference frame to a new coordinate system, generated using a plugin.
We will cover two use cases:

1. **Transforming manually provided points** into the new reference frame.
2. **Transforming cell centroids extracted from a segmentation mask** into the new reference frame.

Each section will guide you through the process, from loading data to saving transformed coordinates.
"""))


# 📌 Coordinate Transformation Notebook

This notebook helps users transform coordinates from a global reference frame to a new coordinate system, generated using a plugin.
We will cover two use cases:

1. **Transforming manually provided points** into the new reference frame.
2. **Transforming cell centroids extracted from a segmentation mask** into the new reference frame.

Each section will guide you through the process, from loading data to saving transformed coordinates.


In [2]:
# Function to load the coordinate system from CSV
def load_coordinate_system(csv_path):
    """
    Loads the new coordinate system's origin and basis vectors from a CSV file.
    The CSV should have three rows: the first for the origin, and the next two for the basis vectors.
    """
    df = pd.read_csv(csv_path, header=None)
    origin = list(map(float, df[0][1].strip("[]").split()))
    u, v, w = list(map(float, df[1][1].strip("[]").split())), list(map(float, df[2][1].strip("[]").split())),                       list(map(float, df[3][1].strip("[]").split()))

    return origin, u, v, w

# Function to transform points to the new coordinate system
def transform_points(points, origin, u, v, w):
    """
    Transforms a set of points into the new coordinate system.
    """
    transformation_matrix = np.column_stack((u, v, w))
    transformed_points = np.linalg.solve(transformation_matrix, (points - origin).T).T
    return transformed_points


In [26]:

# Section 1: Transforming user-provided points
display(Markdown("""
## 🟢 Section 1: Transforming User-Provided Points

For each image, we have already created a **new reference frame**.
Now, for each image, we also have a **list of points** whose coordinates are given in the original reference frame.

### **Goal of this section**
- We want to **convert these points** into the new reference frame assigned to each image.
- Each image has a corresponding **reference frame** stored in a CSV file (`image_name_repere.csv`).
- Each image also has an associated **list of points** stored in a CSV file (`image_name_points.csv`).

### **Processing workflow**
1. **Detect all point files** (`image_name_points.csv`) in the folder.
2. **Find the corresponding reference frame** (`image_name_repere.csv`).
3. **Transform each point's coordinates** using the new reference frame.
4. **Save the transformed coordinates (`X_new, Y_new, Z_new`)** in the same CSV file alongside the original coordinates.

This ensures that all manually provided points are expressed consistently in the new reference system, making it easier to compare and analyze spatial data across images.
"""))



## 🟢 Section 1: Transforming User-Provided Points

For each image, we have already created a **new reference frame**.
Now, for each image, we also have a **list of points** whose coordinates are given in the original reference frame.

### **Goal of this section**
- We want to **convert these points** into the new reference frame assigned to each image.
- Each image has a corresponding **reference frame** stored in a CSV file (`image_name_repere.csv`).
- Each image also has an associated **list of points** stored in a CSV file (`image_name_points.csv`).

### **Processing workflow**
1. **Detect all point files** (`image_name_points.csv`) in the folder.
2. **Find the corresponding reference frame** (`image_name_repere.csv`).
3. **Transform each point's coordinates** using the new reference frame.
4. **Save the transformed coordinates (`X_new, Y_new, Z_new`)** in the same CSV file alongside the original coordinates.

This ensures that all manually provided points are expressed consistently in the new reference system, making it easier to compare and analyze spatial data across images.


In [23]:
def transform_points_csv(input_folder):
    """
    Process all CSV files containing point coordinates in the input folder and transform them.
    - Detects corresponding reference CSV files.
    - Updates the original CSV with transformed coordinates.

    Parameters:
    - input_folder: Path to the dataset containing point CSVs and reference coordinate CSVs.
    """
    files = os.listdir(input_folder)
    point_files = [f for f in files if f.endswith("_points.csv")]

    for point_file in point_files:
        base_name = point_file.replace("_points.csv", "")
        points_csv_path = os.path.join(input_folder, point_file)
        ref_csv_path = os.path.join(input_folder, base_name + ".csv")

        # Ensure the reference CSV exists
        if not os.path.exists(ref_csv_path):
            print(f"⚠️ Missing reference file for {base_name}, skipping.")
            continue

        # Load the reference coordinate system
        origin, u, v, w = load_coordinate_system(ref_csv_path)

        # Load the points CSV
        df = pd.read_csv(points_csv_path)

        # Ensure the required columns exist
        if not {"Z", "Y", "X"}.issubset(df.columns):
            print(f"⚠️ Skipping {point_file}: Missing required columns (Z, Y, X).")
            continue

        points = df[["Z", "Y", "X"]].values

        # Transform the points
        transformed_points = transform_points(points, origin, u, v, w)

        # Add new columns to the original DataFrame
        df[["X_new", "Y_new", "Z_new"]] = transformed_points

        # Save the updated CSV
        df.to_csv(points_csv_path, index=False)
        print(f"✅ Transformed points saved in {points_csv_path}")

In [20]:
# Define the input directory
input_folder = "path/to/your/dataset"  # Change this to the actual dataset path

# Run the transformation process
transform_points_csv(input_folder)

FileNotFoundError: [Errno 2] No such file or directory: 'path/to/your/dataset'

In [10]:

display(Markdown("""
## 🔵 Section 2: Transforming Cell Centroids from a Segmentation Mask

Once an orthonormal reference frame has been created for the images in the folder,
this section allows us to compute the transformed coordinates of cell centroids
for each image that has an associated segmentation mask (the files must be named as follow : image_name_mask.tif).

### How it works:
1. **Extract centroids**: Identify the centroids of segmented regions using `regionprops`.
2. **Apply transformation**: Convert these centroids into the new reference frame.
3. **Save the results**: Store the transformed coordinates in a structured output CSV for each image.

By doing this, we ensure that the spatial information of segmented cells is consistently
aligned with the reference coordinate system, facilitating further quantitative analysis.
"""))



## 🔵 Section 2: Transforming Cell Centroids from a Segmentation Mask

Once an orthonormal reference frame has been created for the images in the folder,
this section allows us to compute the transformed coordinates of cell centroids
for each image that has an associated segmentation mask.

### How it works:
1. **Extract centroids**: Identify the centroids of segmented regions using `regionprops`.
2. **Apply transformation**: Convert these centroids into the new reference frame.
3. **Save the results**: Store the transformed coordinates in a structured output CSV for each image.

By doing this, we ensure that the spatial information of segmented cells is consistently
aligned with the reference coordinate system, facilitating further quantitative analysis.


In [11]:
def process_segmentation_folder(input_folder):
    """
    Process all segmentation masks in the input folder and transform the centroids.
    - Detects corresponding reference CSV files.
    - Saves the transformed centroids in a structured folder.

    Parameters:
    - input_folder: Path to the dataset containing images, masks, and reference CSVs.
    """

    # Get all segmentation masks in the folder
    mask_files = [f for f in os.listdir(input_folder) if f.endswith("_mask.tif")]

    for mask_file in mask_files:
        base_name = mask_file.replace("_mask.tif", "")
        mask_path = os.path.join(input_folder, mask_file)
        ref_csv_path = os.path.join(input_folder, base_name + ".csv")

        # Ensure the reference CSV exists
        if not os.path.exists(ref_csv_path):
            print(f"⚠️ Missing reference file for {base_name}, skipping.")
            continue

        # Load segmentation mask
        mask = skimage.io.imread(mask_path)

        # Extract cell centroids using regionprops
        props = skimage.measure.regionprops(mask)
        centroids = np.array([prop.centroid for prop in props])

        # Load the reference coordinate system
        origin, u, v, w = load_coordinate_system(ref_csv_path)

        # Transform centroids
        transformed_centroids = transform_points(centroids, origin, u, v, w)

        # Save the transformed centroids
        output_csv_path = os.path.join(input_folder, base_name + "_cells_coordinates.csv")
        pd.DataFrame(transformed_centroids, columns=["X_new", "Y_new", "Z_new"]).to_csv(output_csv_path, index=False)

        print(f"✅ Transformed centroids saved: {output_csv_path}")

In [ ]:
# Define input directory
input_folder = "path/to/your/dataset"  # Change this to the actual dataset path

# Run the processing function
process_segmentation_folder(input_folder)

In [ ]:

display(Markdown("""
## 🔵 Section 3: Transforming TrackMate Tracking results

Once an orthonormal reference frame has been created for the images in the folder,
this section allows us to compute the transformed coordinates of cell centroids
for each image that has an associated segmentation mask (the files must be named as follow : image_name_mask.tif).

### How it works:
1. **Extract centroids**: Identify the centroids of segmented regions using `regionprops`.
2. **Apply transformation**: Convert these centroids into the new reference frame.
3. **Save the results**: Store the transformed coordinates in a structured output CSV for each image.

By doing this, we ensure that the spatial information of segmented cells is consistently
aligned with the reference coordinate system, facilitating further quantitative analysis.
"""))

In [9]:
def transform_trackmate_csv(input_folder, format="csv"):
    """
    Process TrackMate CSV file containing spots coordinates across time.

    Parameters:
    - input_folder: Path to the dataset containing point CSVs and reference coordinate CSVs.
    """
    files = os.listdir(input_folder)
    if format == "csv":
        point_files = [f for f in files if f.endswith("_spots.csv")]

        for point_file in point_files:
            base_name = point_file.replace("_spots.csv", "")
            tracks_csv_path = os.path.join(input_folder, point_file)
            ref_csv_path = os.path.join(input_folder, base_name + ".csv")

            # Ensure the reference CSV exists
            if not os.path.exists(os.path.join(input_folder, ref_csv_path)):
                print(f"⚠️ Missing reference file for {base_name}, skipping.")
                continue

            # Load the reference coordinate system
            origin, u, v, w = load_coordinate_system(ref_csv_path)

            # Load the points CSV
            df = pd.read_csv(tracks_csv_path)

            # Extract point coordinates (Z, Y, X) and reorder them to (X, Y, Z) for consistency
            points = df[["POSITION_Z", "POSITION_Y", "POSITION_X"]].values[3:, :].astype("float32")

            # Transform the points
            transformed_points = transform_points(points, origin, u, v, w)

            header = np.array([["Z", "Y", "X"], ["Z", "Y", "X"], ["(micron)", "(micron)", "(micron)"]])

            res = np.vstack((header, transformed_points))

            # Add new columns to the original DataFrame
            df[["NEW_POSITION_X", "NEW_POSITION_Y", "NEW_POSITION_Z"]] = np.vstack((header, transformed_points))

            # Save the updated CSV
            df.to_csv(tracks_csv_path, index=False)
            print(f"✅ Transformed points saved in {tracks_csv_path}")
    else:
        point_files = [f for f in files if f.endswith(".xml")]


In [10]:
# Define input directory
input_folder = "path/to/your/dataset"  # Change this to the actual dataset path
input_folder = "/home/clement/Images/film_vincent"  # Change this to the actual dataset path

# Define whether the results were saved as csv or xml
format = "csv"
transform_trackmate_csv(input_folder, format)

✅ Transformed points saved in /home/clement/Images/film_vincent/Film-4D-6temps-Vincent_t2_spots.csv
